# Receitas da Prefeitura de Palmas
### Pipeline de dados · módulo `sgreceitas` · Portal NUCLEOGOV

> Coleta de **20/10/2026** · 180.976 registros embutidos neste notebook.
> Para rodar: **Ambiente de execução → Executar tudo**. Não é preciso enviar nenhum arquivo.

| Etapa | O que acontece |
|---|---|
| 1 · Dados | Carrega os dados embutidos e confere o teste do item 3.2 do manual |
| 2 · Limpeza | Padroniza nomes, remove duplicatas e organiza os códigos de receita |
| 3 · Exportação | CSV tratado, relatório do ETL e README da coleta |

In [ ]:
#@title Identificação da equipe
integrante_1 = "Davi Pereira" #@param {type:"string"}
integrante_2 = "Laura Nicolle" #@param {type:"string"}
integrante_3 = "Márcio Araújo" #@param {type:"string"}
integrante_4 = "Matheus Felipe" #@param {type:"string"}
integrante_5 = "Marcus Vinicius" #@param {type:"string"}
data_coleta = "2026-10-20" #@param {type:"date"}
timeout_confirmado = False #@param {type:"boolean"}

integrantes = [n for n in (integrante_1, integrante_2, integrante_3, integrante_4, integrante_5) if n.strip()]
print("Equipe:", ", ".join(integrantes) if integrantes else "-", "| Coleta:", data_coleta)

---
## 1 · Dados
**Fonte:** portal novo (NUCLEOGOV), módulo `sgreceitas`. Foi escolhido por trazer dados atualizados (2018 a 10/2026) com HTTPS válido; a API antiga está congelada em jun/2021. As duas APIs não foram combinadas, pois não compartilham identificadores nem esquema.

In [ ]:
#@title Dados embutidos (sgreceitas_2026-10-20.csv, compactado)
DADOS_B64 = """
...
"""

In [ ]:
import base64, io, lzma, re
import numpy as np
import pandas as pd
from IPython.display import HTML, display

def cartoes(itens):
    """Mostra uma linha de cartões de resumo: [(valor, rótulo), ...]."""
    html = "".join(
        f"<div style='flex:1;min-width:150px;padding:14px 16px;border-radius:10px;background:#f3f6fb;"
        f"border-left:5px solid #0072B2;font-family:sans-serif'>"
        f"<div style='font-size:22px;font-weight:700;color:#0072B2'>{v}</div>"
        f"<div style='font-size:12px;color:#57606a;margin-top:2px'>{r}</div></div>" for v, r in itens)
    display(HTML(f"<div style='display:flex;gap:12px;flex-wrap:wrap;margin:8px 0'>{html}</div>"))

raw = pd.read_csv(io.BytesIO(lzma.decompress(base64.b64decode("".join(DADOS_B64.split())))),
                  dtype=str, encoding="utf-8-sig")

vazias = [c for c in raw if raw[c].isna().all()]
cartoes([(f"{len(raw):,}".replace(",", "."), "linhas carregadas"),
         (raw.shape[1], "colunas"),
         (raw["orgao_nome"].nunique(), "nomes de órgão (brutos)"),
         (len(vazias), "colunas 100% vazias")])
print("Colunas vazias:", ", ".join(vazias))
raw.head()

In [ ]:
#@title Verificação do teste do item 3.2 (offline, usa os dados embutidos)
ESPERADO = ["total", "orgao_nome", "unidade_nome", "codigo", "orgao", "ano", "mes", "descricao",
            "valor_orcado", "valor_arrecado_mes", "valor_arrecado_periodo", "covid", "unidade_id", "codigo_original"]
total_api = int(raw["total"].iloc[0])
cartoes([(f"{total_api:,}".replace(",", "."), "total informado pela API"),
         ("Sim" if total_api == len(raw) else "Não", "registros coletados = total"),
         ("Sim" if list(raw.columns) == ESPERADO else "Não", "14 campos iguais ao manual")])
print("Divergência a registrar no relatório: campos 100% nulos ->", ", ".join(vazias))

In [ ]:
#@title Teste ao vivo do item 3.2 (opcional: faz 1 requisição ao portal)
executar_teste = False #@param {type:"boolean"}

if executar_teste:
    import json, time, requests
    from urllib.robotparser import RobotFileParser

    BASE = "https://acessoainformacao.palmas.to.gov.br"
    NAVEGADOR = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                 "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
    CAMPOS_MANUAL = ["total", "orgao_nome", "unidade_nome", "codigo", "orgao", "ano", "mes", "descricao",
                     "valor_orcado", "valor_arrecado_mes", "valor_arrecado_periodo", "covid", "unidade_id",
                     "codigo_original"]

    # regra 2 do manual: verificar o robots.txt antes (com o mesmo User-Agent da coleta)
    r = requests.get(f"{BASE}/robots.txt", headers={"User-Agent": NAVEGADOR}, timeout=60)
    liberado = False
    if r.status_code == 200:
        rp = RobotFileParser()
        rp.parse(r.text.splitlines())
        liberado = rp.can_fetch(NAVEGADOR, f"{BASE}/api")
        print("robots.txt (HTTP 200): acesso a /api permitido:", liberado)
    else:
        print(f"robots.txt respondeu HTTP {r.status_code}: não foi possível verificar a permissão.")

    if not liberado:
        print("Teste interrompido. Use o log da coleta (robots.txt consultado em 20/10/2026) como evidência.")
    else:
        sessao = requests.Session()
        sessao.get(f"{BASE}/cidadao/transparencia/sgreceitas", headers={"User-Agent": NAVEGADOR}, timeout=60)
        time.sleep(0.5)

        consulta = {"k1": {"order": {}, "limit": "0, 1", "acao": "sgreceitas/listar"}}
        resposta = sessao.post(
            f"{BASE}/api",
            data={"multi_request": "true", "params": json.dumps(consulta)},
            headers={"User-Agent": NAVEGADOR, "X-Requested-With": "XMLHttpRequest",
                     "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
                     "Referer": f"{BASE}/cidadao/transparencia/sgreceitas", "Origin": BASE},
            timeout=60)

        dados = resposta.json()["k1"]
        campos = list(dados["dados"][0].keys())
        print("total de registros:", dados["total"])
        print("campos disponíveis:", campos)
        print("Mesmos campos do manual:", campos == CAMPOS_MANUAL)
        if "raw" in globals():
            print("Total na coleta embutida:", int(raw["total"].iloc[0]),
                  "(o total muda com o tempo, por isso a data da coleta é registrada)")
else:
    print("Desativado. Marque 'executar_teste' para repetir o teste do manual (1 requisição, com pausa e timeout).")

---
## 2 · Limpeza (ETL)

In [ ]:
# ---------- colunas e tipos ----------
df = raw.drop(columns=vazias + ["total"]).copy()
for c in ("valor_orcado", "valor_arrecado_mes", "valor_arrecado_periodo"):
    df[c] = pd.to_numeric(df[c])
for c in ("ano", "mes", "orgao"):
    df[c] = df[c].astype(int)
df["data_ref"] = pd.to_datetime(dict(year=df.ano, month=df.mes, day=1))
df["covid"] = df["covid"] == "1"

# ---------- nomes dos órgãos ----------
MINUSCULAS = {"de", "da", "do", "das", "dos", "e", "à", "a", "para", "em", "no", "na"}
ACENTOS = {"AGENCIA": "AGÊNCIA", "REGULACAO": "REGULAÇÃO", "FISCALIZACAO": "FISCALIZAÇÃO",
           "SERVICOS": "SERVIÇOS", "PUBLICOS": "PÚBLICOS", "PREVIDENCIA": "PREVIDÊNCIA",
           "PREVIDENCIARIO": "PREVIDENCIÁRIO", "SEGURANCA": "SEGURANÇA", "PROTECAO": "PROTEÇÃO"}
CORRECOES = {"DOS DIREITO DA MULHER": "DOS DIREITOS DA MULHER"}

def limpar_nome(texto):
    s = " ".join(str(texto).upper().split())
    for errado, certo in CORRECOES.items():
        s = s.replace(errado, certo)
    partes, inicio = [], True
    for tok in re.findall(r"[^\W\d_]+|\d+|\W+", s):
        if tok[0].isalpha():
            tok = ACENTOS.get(tok, tok)
            tok = tok.lower() if (tok.lower() in MINUSCULAS and not inicio) else tok.capitalize()
            inicio = False
        partes.append(tok)
    return "".join(partes)

mapa = {n: limpar_nome(n) for n in df["orgao_nome"].unique()}
df["orgao_nome"] = df["orgao_nome"].map(mapa)
df["orgao_tipo"] = df["orgao_nome"].str.split().str[0].where(
    lambda s: s.isin(["Secretaria", "Fundo", "Fundação", "Instituto", "Agência", "Procuradoria", "Tesouro"]), "Outro")
assert df.groupby("orgao")["orgao_nome"].nunique().max() == 1

# ---------- descrição: espaços e grafia única ----------
df["descricao"] = df["descricao"].str.split().str.join(" ").fillna("(sem descrição)")
grafia = {}
for texto in df["descricao"].value_counts().index:          # mais frequente primeiro
    grafia.setdefault(texto.casefold(), texto)
df["descricao"] = df["descricao"].map(lambda t: grafia[t.casefold()])

# ---------- duplicatas (mesmo órgão, código, ano e mês; fica o código_original mais curto) ----------
chave = ["orgao", "codigo", "ano", "mes"]
df = (df.assign(_t=df["codigo_original"].str.len()).sort_values(chave + ["_t"])
        .drop_duplicates(chave).drop(columns="_t"))

# ---------- estrutura do código de receita ----------
def nivel(codigo):
    if "." not in codigo:
        return np.nan
    return max(i for i, seg in enumerate(codigo.split("."), 1) if set(seg) != {"0"})

df["hierarquico"] = df["codigo"].str.contains(".", regex=False)
df["nivel"] = df["codigo"].map(nivel)
df["categoria"] = df["codigo"].str[0].where(df["hierarquico"])
df = df.sort_values(["data_ref", "orgao_nome", "codigo"]).reset_index(drop=True)

# ---------- resumo ----------
relatorio = {
    "linhas_brutas": len(raw),
    "linhas_finais": len(df),
    "duplicatas_removidas": len(raw) - len(df),
    "colunas_removidas": ", ".join(vazias + ["total"]),
    "nomes_de_orgao_padronizados": sum(a != b for a, b in mapa.items()),
    "descricoes_sem_texto": int((raw["descricao"].isna()).sum()),
    "valores_negativos_de_arrecadacao": int((df["valor_arrecado_mes"] < 0).sum()),
}
cartoes([(f"{relatorio['linhas_finais']:,}".replace(",", "."), "linhas após a limpeza"),
         (f"{relatorio['duplicatas_removidas']:,}".replace(",", "."), "duplicatas removidas"),
         (relatorio["nomes_de_orgao_padronizados"], "nomes de órgão padronizados"),
         (relatorio["valores_negativos_de_arrecadacao"], "valores negativos (estornos, mantidos)")])

In [ ]:
#@title Conferência: nomes de órgão antes e depois
alterados = pd.DataFrame({"antes": list(mapa), "depois": list(mapa.values())}).query("antes != depois")
pd.set_option("display.max_colwidth", 90)
display(alterados.sort_values("depois").head(12).reset_index(drop=True)
        .style.set_table_styles([{"selector": "th", "props": [("background", "#0072B2"), ("color", "white")]}]))
print(f"Mostrando 12 de {len(alterados)} nomes alterados.")

---
## 3 · Exportação

In [ ]:
#@title Exportar arquivos (CSV tratado, relatório do ETL e README)
from datetime import date
try:
    from google.colab import files
except ImportError:
    files = None

def texto_readme(s):
    p = lambda v: f"{v:,}".replace(",", ".")
    timeout = ("todas as requisições do coletor usam `timeout` (confirmado pela equipe)" if timeout_confirmado
               else "a confirmar no código do coletor")
    linhas = [
        "# Coleta de dados: Receitas (sgreceitas), Prefeitura de Palmas", "",
        f"**Data da coleta:** {data_coleta}",
        f"**Equipe:** {', '.join(integrantes) if integrantes else '-'}", "",
        "## Fonte e justificativa",
        "- Portal novo (NUCLEOGOV): `https://acessoainformacao.palmas.to.gov.br/api`, módulo `sgreceitas/listar` (POST com formulário).",
        f"- Escolhido por trazer dados atualizados ({df.data_ref.min():%m/%Y} a {df.data_ref.max():%m/%Y}) com HTTPS válido; "
        "a API antiga está congelada em jun/2021. As duas APIs não foram combinadas, pois não compartilham identificadores nem esquema.", "",
        "## Arquivos",
        f"- `raw_sgreceitas_{data_coleta}.jsonl`: resposta bruta da API.",
        f"- `sgreceitas_{data_coleta}.csv`: tabela coletada. `coleta_sgreceitas_{data_coleta}.log`: registro da coleta.",
        f"- `receitas_tratado_{s}.csv`: dados após o ETL.", "",
        "## Conduta de coleta (seção 2 do manual)",
        "- Pausa entre requisições: 0,5 s (conforme o log da coleta).",
        "- robots.txt consultado antes da coleta; acesso a `/api` permitido (log).",
        f"- Sem automação recorrente: coleta única em {data_coleta}, executada manualmente em 2 etapas "
        "(teste de 1.000 registros e retomada até o fim, cerca de 8 minutos).",
        "- Data da coleta registrada no nome dos arquivos e neste README.",
        f"- Dados pessoais: nenhum. Os campos são classificação orçamentária e valores agregados ({', '.join(df.columns[:6])}, ...).",
        f"- Timeout em toda requisição: {timeout}.", "",
        "## Primeiro teste (item 3.2)",
        f"- Total informado pela API: {p(total_api)}; registros coletados: {p(len(raw))}.",
        f"- Campos: {'idênticos aos do manual' if list(raw.columns) == ESPERADO else 'DIFERENTES do manual'} (14 campos).",
        "- O primeiro teste da equipe coletou 1.000 registros (log). O teste de 1 registro do manual está na célula opcional do notebook.",
        f"- Divergência encontrada: {', '.join(vazias)} vieram 100% nulos, embora o manual os liste entre os campos disponíveis.", "",
        "## Resultado do ETL",
    ] + [f"- {k}: {v}" for k, v in relatorio.items()]
    return "\n".join(linhas) + "\n"

s = (data_coleta or date.today().isoformat()).replace("-", "")
csv, txt, readme = f"receitas_tratado_{s}.csv", f"relatorio_etl_{s}.txt", f"README_{s}.md"

df.to_csv(csv, index=False, encoding="utf-8")
with open(txt, "w", encoding="utf-8") as f:
    f.write(f"Origem: sgreceitas_2026-10-20.csv (embutido) | Coleta: {data_coleta}\n")
    f.write(f"Equipe: {', '.join(integrantes) if integrantes else '-'}\n\n")
    f.writelines(f"{k}: {v}\n" for k, v in relatorio.items())
with open(readme, "w", encoding="utf-8") as f:
    f.write(texto_readme(s))

print("Gerados:", csv, txt, readme, sep="\n  ")
if files:
    for arq in (csv, txt, readme):
        files.download(arq)